# Building Multimodal RAG Application — Part 7: LangChain Pipeline

This Colab-ready companion notebook composes retrieval, prompt processing, and LVLM inference with current LangChain Core runnables. It follows the article's `RunnableParallel`, `RunnablePassthrough`, and `RunnableLambda` architecture and returns both the answer and the retrieved frame.

> **Runtime note:** text–image retrieval uses CLIP and generation defaults to the compact open `llava-hf/llava-interleave-qwen-0.5b-hf` checkpoint from the official Transformers image-text-to-text guide. Change `LVLM_MODEL_ID` to `llava-hf/llava-1.5-7b-hf` only on a runtime with enough GPU memory.

## 1. Install dependencies

In [ ]:
!pip install -q "langchain-core>=1.0" "sentence-transformers>=3.0" "transformers>=4.52" accelerate bitsandbytes pillow requests

## 2. Imports and configuration

In [ ]:
from io import BytesIO

import requests
import torch
from IPython.display import display
from PIL import Image
from langchain_core.runnables import (
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
)
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline

CLIP_MODEL_ID = "clip-ViT-B-32"
LVLM_MODEL_ID = "llava-hf/llava-interleave-qwen-0.5b-hf"
RUN_LVLM = True

## 3. Prepare a small multimodal collection

The records stand in for video frames and neighboring transcripts returned by the preprocessing stage in Part 4. In a production application, these records would live in the vector store built in Part 5.

In [ ]:
records = [
    {
        "id": "skateboard",
        "image_url": "https://farm4.staticflickr.com/3300/3497460990_11dfb95dd1_z.jpg",
        "transcript": "A skateboarder performs an inverted trick in a concrete skate park.",
    },
    {
        "id": "diagram",
        "image_url": (
            "https://huggingface.co/datasets/huggingface/documentation-images/"
            "resolve/main/transformers/tasks/ai2d-demo.jpg"
        ),
        "transcript": "A scientific diagram with numbered labels and arrows.",
    },
]

def load_image(url):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return Image.open(BytesIO(response.content)).convert("RGB")

for record in records:
    record["image"] = load_image(record["image_url"])
    display(record["image"])

## 4. Build the retrieval module

In [ ]:
clip_model = SentenceTransformer(CLIP_MODEL_ID)
image_embeddings = clip_model.encode(
    [record["image"] for record in records],
    convert_to_tensor=True,
    normalize_embeddings=True,
)

def retrieve_frame(query):
    query_embedding = clip_model.encode(
        query, convert_to_tensor=True, normalize_embeddings=True
    )
    scores = util.cos_sim(query_embedding, image_embeddings)[0]
    best_index = int(torch.argmax(scores))
    result = dict(records[best_index])
    result["score"] = float(scores[best_index])
    return result

retriever_module = RunnableLambda(retrieve_frame)

## 5. Build prompt processing and LVLM inference modules

In [ ]:
def prepare_lvlm_input(inputs):
    retrieved = inputs["retrieved_results"]
    prompt = (
        "Use the retrieved image and evidence description to answer the question.\n"
        f"Evidence description: {retrieved['transcript']}\n"
        f"Question: {inputs['user_query']}\n"
        "Respond with one complete, specific sentence. Do not answer with a fragment."
    )
    return {
        "image": retrieved["image"],
        "image_url": retrieved["image_url"],
        "retrieval_score": retrieved["score"],
        "prompt": prompt,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": retrieved["image_url"]},
                    {"type": "text", "text": prompt},
                ],
            }
        ],
    }

prompt_processing_module = RunnableLambda(prepare_lvlm_input)

In [ ]:
generator = None
if RUN_LVLM:
    generator = pipeline(
        task="image-text-to-text",
        model=LVLM_MODEL_ID,
        device_map="auto",
        dtype=(torch.float16 if torch.cuda.is_available() else torch.float32),
    )
else:
    print("LLaVA loading skipped. The chain will return a prompt preview.")

def run_lvlm(payload):
    if generator is None:
        return (
            "[LVLM preview] "
            + payload["prompt"]
            + f"\nRetrieved image: {payload['image_url']}"
        )
    result = generator(
        text=payload["messages"],
        max_new_tokens=80,
        do_sample=False,
        return_full_text=False,
    )
    generated = result[0]["generated_text"]
    if isinstance(generated, list):
        return generated[-1]["content"]
    return generated

lvlm_inference_module = RunnableLambda(run_lvlm)

## 6. Compose and invoke the basic multimodal RAG chain

In [ ]:
mm_rag_chain = (
    RunnableParallel(
        {
            "retrieved_results": retriever_module,
            "user_query": RunnablePassthrough(),
        }
    )
    | prompt_processing_module
    | lvlm_inference_module
)

query = "What is likely to happen next in the skateboarding scene?"
answer = mm_rag_chain.invoke(query)
print(f"USER Query: {query}\n")
print(f"MM-RAG Response: {answer}")

## 7. Return the answer and retrieved frame in parallel

A second `RunnableParallel` preserves the prepared LVLM input while producing the answer, making the evidence visible to the application and its users.

In [ ]:
mm_rag_chain_with_retrieved_image = (
    RunnableParallel(
        {
            "retrieved_results": retriever_module,
            "user_query": RunnablePassthrough(),
        }
    )
    | prompt_processing_module
    | RunnableParallel(
        {
            "final_text_output": lvlm_inference_module,
            "input_to_lvlm": RunnablePassthrough(),
        }
    )
)

query = "What kind of image contains numbered labels?"
response = mm_rag_chain_with_retrieved_image.invoke(query)

print(f"USER Query: {query}\n")
print(f"MM-RAG Response: {response['final_text_output']}\n")
print(f"Retrieval score: {response['input_to_lvlm']['retrieval_score']:.3f}")
display(response["input_to_lvlm"]["image"])

## 8. Next step

Part 8 assembles the complete workflow: video preprocessing, joint embeddings, vector storage, retrieval, prompt construction, LVLM inference, and evidence-aware responses.